# signals.ipynb — Construcción de señales y retornos de estrategia

**Propósito:** Toma `prices_adjusted.csv` y genera series de retornos diarios por estrategia, guardadas en `data/signals/daily_returns_XXXX.csv`.

**Flujo:**
1. Cargar precios ajustados + configuración
2. Funciones de señal: MA gradual, binaria, position sizing
3. Aplicar señales a cada estrategia
4. Guardar `daily_returns_XXXX.csv` en `data/signals/`

**Pre-requisito:** `research.ipynb` ya corrido.  
**Output:** `data/signals/daily_returns_STRAT001.csv`, `...002.csv`, etc.  
**Referencia:** MScFE SKILL §5, §6, López de Prado §4


In [1]:
# ── 0. PATHS ──────────────────────────────────────────────────────────────────
from pathlib import Path

def find_repo_root(marker: str = 'pyproject.toml') -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {current}")


BASE    = find_repo_root()
CONFIG  = BASE / 'config'
DATA    = BASE / 'data'
SIGNALS = DATA / 'signals'
SIGNALS.mkdir(parents=True, exist_ok=True)

print('BASE:   ', BASE, BASE.exists())
print('prices_adjusted.csv:', (DATA / 'prices_adjusted.csv').exists())
print('signals/ dir:', SIGNALS.exists())

BASE:    C:\Users\Sebastián\portfolio_quant True
prices_adjusted.csv: True
signals/ dir: True


In [2]:
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt

TRADING_DAYS = 252

try:
    cr_df     = pd.read_csv(CONFIG / 'costs_rates.csv')
    rf_annual = float(cr_df.loc[cr_df['parameter'] == 'risk_free_rate', 'value'].iloc[0])
except Exception:
    rf_annual = 0.02
    print('[warn] rf_annual=0.02 por defecto')

try:
    rules          = yaml.safe_load((CONFIG / 'rules.yaml').read_text())
    ma_length      = rules['signals']['primary']['ma_length']
    apply_fraction = rules['signals']['primary']['apply_fraction']
except Exception:
    ma_length, apply_fraction = 15, 1/6
    print('[warn] ma_length=15, apply_fraction=1/6 por defecto')

print(f'rf_annual={rf_annual:.2%} | ma_length={ma_length} | apply_fraction={apply_fraction:.4f}')

rf_annual=2.00% | ma_length=15 | apply_fraction=0.1667


In [3]:
prices_adjusted = pd.read_csv(
    DATA / 'prices_adjusted.csv',
    parse_dates=['date'],
    index_col='date'
).sort_index()

returns_daily = prices_adjusted.pct_change().dropna(how='all')

print(f'Panel: {prices_adjusted.shape}')
print(f'Período: {prices_adjusted.index.min().date()} → {prices_adjusted.index.max().date()}')
print(f'Columnas disponibles: {list(prices_adjusted.columns)}')

Panel: (2512, 17)
Período: 2016-06-27 → 2026-06-24
Columnas disponibles: ['AGG', 'BOXX', 'EEM', 'EFA', 'EMB', 'GLD', 'HYG', 'IEF', 'LQD', 'PFF', 'SHY', 'SLV', 'SPY', 'TIP', 'TLT', 'VEA', 'VNQ']


## Funciones de señal

Todas las señales usan datos de **t-1** para la posición en **t** (sin look-ahead bias).  
Ref: MScFE SKILL §8.2 (Trap 5)


In [4]:
# ── FUNCIONES DE SEÑAL ────────────────────────────────────────────────────────

def signal_ma_binary(price_series: pd.Series, ma_window: int = 15) -> pd.Series:
    """Señal binaria: 1 si precio > MA, 0 si no. Desplazada 1 día (sin look-ahead)."""
    ma         = price_series.rolling(ma_window).mean()
    signal_raw = (price_series > ma).astype(float)
    return signal_raw.shift(1).fillna(0.0)


def signal_ma_gradual(price_series: pd.Series, ma_window: int = 15,
                      step: float = 1/6, rebal_days_signal: int = 14) -> pd.Series:
    """Señal gradual: cambia posición en `step` cada `rebal_days_signal` días. Desplazada 1 día."""
    ma             = price_series.rolling(ma_window).mean()
    below          = (price_series < ma).fillna(False)
    hedge_fraction = 0.0
    position_list  = []
    last_rebal_idx = None

    for i, (dt, is_below) in enumerate(below.items()):
        if last_rebal_idx is None or (i - last_rebal_idx) >= rebal_days_signal:
            if is_below:
                hedge_fraction = min(1.0, hedge_fraction + step)
            else:
                hedge_fraction = max(0.0, hedge_fraction - step)
            last_rebal_idx = i
        position_list.append(1.0 - hedge_fraction)

    position_series = pd.Series(position_list, index=price_series.index)
    return position_series.shift(1).fillna(1.0)


def compute_position_sizing_equal(ticker_list: list, prices_panel: pd.DataFrame) -> pd.DataFrame:
    """Pesos iguales entre tickers disponibles en cada fecha. Ref: MScFE SKILL §3.1."""
    available  = [t for t in ticker_list if t in prices_panel.columns]
    valid_mask = prices_panel[available].notna()
    row_counts = valid_mask.sum(axis=1).replace(0, np.nan)
    return valid_mask.div(row_counts, axis=0).fillna(0.0)


def signal_always_long(price_series: pd.Series, **kwargs) -> pd.Series:
    """Señal constante: posición larga plena (sin timing). Usada por benchmarks."""
    return pd.Series(1.0, index=price_series.index)


print('signal_ma_binary | signal_ma_gradual | compute_position_sizing_equal | signal_always_long')

signal_ma_binary | signal_ma_gradual | compute_position_sizing_equal | signal_always_long


In [5]:
# ── FUNCIÓN CORE: CONSTRUIR RETORNOS DE ESTRATEGIA ────────────────────────────

def build_strategy_returns(
    system_tickers: list,
    hedge_map:      dict,
    weights_df:     pd.DataFrame,
    signal_fn,
    collateral_tickers: list = ['BOXX', 'SHY'],
    costs_bps:  float = 10.0,
    rebal_days: int   = 14,
    **signal_kwargs
) -> pd.Series:
    """
    Construye serie de retornos diarios para una estrategia.
    Señal en t-1, costos en bps cada rebal_days días.

    Args:
        system_tickers     (list):         Tickers del sistema.
        hedge_map          (dict):         ticker → instrumento hedge o None.
        weights_df         (pd.DataFrame): Pesos por ticker y fecha.
        signal_fn          (callable):     Función de señal (price_series, **kwargs).
        collateral_tickers (list):         Orden de preferencia colateral.
        costs_bps          (float):        Costos transacción en bps.
        rebal_days         (int):          Días entre rebalanceos.
        **signal_kwargs:   Parámetros para signal_fn.

    Returns:
        pd.Series: Retornos diarios de la estrategia.
    """
    collateral_col     = next((t for t in collateral_tickers if t in returns_daily.columns), None)
    returns_collateral = returns_daily[collateral_col] if collateral_col else pd.Series(0.0, index=returns_daily.index)

    tickers_available = [t for t in system_tickers if t in returns_daily.columns]
    if not tickers_available:
        raise ValueError(f'Ningún ticker disponible: {system_tickers}')

    # Señales por ticker (posición larga en [0,1])
    position_signals = {}
    for ticker in tickers_available:
        if ticker in prices_adjusted.columns:
            position_signals[ticker] = signal_fn(prices_adjusted[ticker], **signal_kwargs)
        else:
            position_signals[ticker] = pd.Series(1.0, index=prices_adjusted.index)

    strategy_returns = []
    last_rebal_date  = None
    common_index     = returns_daily[tickers_available].dropna(how='all').index

    for dt in common_index:
        if last_rebal_date is None:
            cost = 0.0
            last_rebal_date = dt
        elif (dt - last_rebal_date).days >= rebal_days:
            cost = costs_bps / 10_000
            last_rebal_date = dt
        else:
            cost = 0.0

        daily_ret = 0.0
        for ticker in tickers_available:
            w_asset   = weights_df.loc[dt, ticker] if dt in weights_df.index and ticker in weights_df.columns else 1.0 / len(tickers_available)
            pos_long  = position_signals[ticker].get(dt, 1.0)
            ret_asset = returns_daily.loc[dt, ticker] if dt in returns_daily.index else 0.0
            ret_coll  = returns_collateral.get(dt, 0.0)
            if pd.isna(ret_asset):
                ret_asset = 0.0
            daily_ret += w_asset * (pos_long * ret_asset + (1.0 - pos_long) * ret_coll)

        strategy_returns.append(daily_ret - cost)

    return pd.Series(strategy_returns, index=common_index, name='daily_return')


print('build_strategy_returns() definida')

build_strategy_returns() definida


## Funciones auxiliares — Dual Momentum (STRAT-005)

`rolling_sharpe_matrix`: Sharpe ratio anualizado rolling por ticker.  
`select_dual_momentum`: Selección cross-sectional top-N + filtro absoluto vs. SHY (momentum dual).  
Ref: MScFE SKILL §6.1, §8.2 (Trap 5)

In [6]:
# ── FUNCIONES AUXILIARES — DUAL MOMENTUM (STRAT-005) ──────────────────────────

def rolling_sharpe_matrix(returns_panel: pd.DataFrame, lookback: int = 21,
                           rf_annual: float = 0.02, periods: int = 252) -> pd.DataFrame:
    """
    Sharpe ratio anualizado rolling por ticker.
    Ref: MScFE SKILL §6.1 (rolling features), §1.3 (annualized_sharpe).
    """
    rf_daily   = rf_annual / periods
    excess     = returns_panel - rf_daily
    roll_mean  = excess.rolling(lookback).mean()
    roll_std   = excess.rolling(lookback).std(ddof=1)
    return (roll_mean / roll_std) * np.sqrt(periods)


def select_dual_momentum(sharpe_panel: pd.DataFrame, rf_ticker: str = 'SHY',
                          n_select: int = 5, rebal_days_signal: int = 21) -> pd.DataFrame:
    """
    Selecciona n_select tickers con mejor Sharpe rolling en cada rebalanceo.
    Momentum dual: slots que no superan al rf_ticker se asignan a rf_ticker.
    Shifted 1 día (sin look-ahead). Ref: MScFE SKILL §8.2 (Trap 5).
    """
    if rf_ticker not in sharpe_panel.columns:
        raise ValueError(f'{rf_ticker} no está en el panel de Sharpe ratios.')

    tickers        = list(sharpe_panel.columns)
    weights_list   = []
    current_w      = pd.Series(0.0, index=tickers)
    last_rebal_idx = None

    for i, (dt, row) in enumerate(sharpe_panel.iterrows()):
        if row.isna().all():
            weights_list.append(current_w.copy())
            continue

        if last_rebal_idx is None or (i - last_rebal_idx) >= rebal_days_signal:
            rf_sharpe = row.get(rf_ticker, -np.inf)
            ranked    = row.dropna().sort_values(ascending=False)
            top_n     = ranked.head(n_select).index.tolist()

            final_selection = [
                t if row[t] > rf_sharpe else rf_ticker
                for t in top_n
            ]

            new_w = pd.Series(0.0, index=tickers)
            for t in final_selection:
                new_w[t] += 1.0 / n_select

            current_w      = new_w
            last_rebal_idx = i

        weights_list.append(current_w.copy())

    weights_df = pd.DataFrame(weights_list, index=sharpe_panel.index)
    return weights_df.shift(1).fillna(0.0)


print('rolling_sharpe_matrix | select_dual_momentum')

rolling_sharpe_matrix | select_dual_momentum


## STRAT-001 — Sistema Return Stacking con Hedge Gradual

Parámetros según `portfolio_strategies.md` y `rules.yaml`.


In [7]:
STRAT001_ID      = 'STRAT001'
strat001_tickers = ['RSSB', 'RSST', 'GLD', 'SLV', 'TIP', 'HYG', 'PFF']
strat001_hedge   = {'RSSB': 'VT', 'RSST': 'SPY', 'GLD': None, 'SLV': None}

strat001_available = [t for t in strat001_tickers if t in prices_adjusted.columns]
print(f'STRAT-001 tickers disponibles: {strat001_available}')
if len(strat001_available) < len(strat001_tickers):
    missing = [t for t in strat001_tickers if t not in prices_adjusted.columns]
    print(f'  [warn] Faltantes: {missing}')

weights_strat001 = compute_position_sizing_equal(strat001_available, prices_adjusted)

returns_strat001 = build_strategy_returns(
    system_tickers=strat001_available,
    hedge_map=strat001_hedge,
    weights_df=weights_strat001,
    signal_fn=signal_ma_gradual,
    collateral_tickers=['BOXX', 'SHY'],
    costs_bps=10.0,
    rebal_days=14,
    ma_window=ma_length,
    step=apply_fraction,
    rebal_days_signal=14,
)

output_path_001 = SIGNALS / f'daily_returns_{STRAT001_ID}.csv'
returns_strat001.to_frame('daily_return').to_csv(output_path_001)
print(f'STRAT-001 guardado: {output_path_001}')
print(f'Período: {returns_strat001.index.min().date()} → {returns_strat001.index.max().date()}')
print(f'Retorno medio diario: {returns_strat001.mean():.4%}')

STRAT-001 tickers disponibles: ['GLD', 'SLV', 'TIP', 'HYG', 'PFF']
  [warn] Faltantes: ['RSSB', 'RSST']


STRAT-001 guardado: C:\Users\Sebastián\portfolio_quant\data\signals\daily_returns_STRAT001.csv
Período: 2016-06-28 → 2026-06-24
Retorno medio diario: 0.0366%


## STRAT-002 — 60/40 SPY/AGG (Benchmark)


In [8]:
STRAT002_ID            = 'STRAT002'
strat002_tickers       = ['SPY', 'AGG']
strat002_weights_fixed = {'SPY': 0.60, 'AGG': 0.40}

strat002_available = [t for t in strat002_tickers if t in returns_daily.columns]
print(f'STRAT-002 tickers disponibles: {strat002_available}')

if len(strat002_available) == 0:
    print('[ERROR] SPY o AGG no disponibles en el panel.')
else:
    weights_strat002 = pd.DataFrame(
        {t: strat002_weights_fixed.get(t, 0.0) for t in strat002_available},
        index=prices_adjusted.index
    )

    returns_strat002 = build_strategy_returns(
        system_tickers=strat002_available,
        hedge_map={},
        weights_df=weights_strat002,
        signal_fn=signal_always_long,
        costs_bps=5.0,
        rebal_days=21,
    )

    output_path_002 = SIGNALS / f'daily_returns_{STRAT002_ID}.csv'
    returns_strat002.to_frame('daily_return').to_csv(output_path_002)
    print(f'STRAT-002 guardado: {output_path_002}')
    print(f'Retorno medio diario: {returns_strat002.mean():.4%}')

STRAT-002 tickers disponibles: ['SPY', 'AGG']
STRAT-002 guardado: C:\Users\Sebastián\portfolio_quant\data\signals\daily_returns_STRAT002.csv
Retorno medio diario: 0.0552%


## STRAT-004 — Permanent Portfolio (Benchmark)

25% SPY / 25% TLT / 25% GLD / 25% SHY. Pesos fijos, sin señal de momentum.

In [9]:
STRAT004_ID            = 'STRAT004'
strat004_tickers       = ['SPY', 'TLT', 'GLD', 'SHY']
strat004_weights_fixed = {'SPY': 0.25, 'TLT': 0.25, 'GLD': 0.25, 'SHY': 0.25}

strat004_available = [t for t in strat004_tickers if t in returns_daily.columns]
print(f'STRAT-004 tickers disponibles: {strat004_available}')
if len(strat004_available) < len(strat004_tickers):
    missing = [t for t in strat004_tickers if t not in returns_daily.columns]
    print(f'  [warn] Faltantes: {missing}')

if len(strat004_available) == 0:
    print('[ERROR] Ningún ticker del Permanent Portfolio disponible en el panel.')
else:
    weights_strat004 = pd.DataFrame(
        {t: strat004_weights_fixed.get(t, 0.0) for t in strat004_available},
        index=prices_adjusted.index
    )

    returns_strat004 = build_strategy_returns(
        system_tickers=strat004_available,
        hedge_map={},
        weights_df=weights_strat004,
        signal_fn=signal_always_long,
        costs_bps=5.0,
        rebal_days=21,
    )

    output_path_004 = SIGNALS / f'daily_returns_{STRAT004_ID}.csv'
    returns_strat004.to_frame('daily_return').to_csv(output_path_004)
    print(f'STRAT-004 guardado: {output_path_004}')
    print(f'Período: {returns_strat004.index.min().date()} → {returns_strat004.index.max().date()}')
    print(f'Retorno medio diario: {returns_strat004.mean():.4%}')

STRAT-004 tickers disponibles: ['SPY', 'TLT', 'GLD', 'SHY']


STRAT-004 guardado: C:\Users\Sebastián\portfolio_quant\data\signals\daily_returns_STRAT004.csv
Período: 2016-06-28 → 2026-06-24
Retorno medio diario: 0.0474%


## STRAT-005 — Dual Momentum Cross-Sectional (Sharpe ranking)

**Lógica:** en cada rebalanceo (21 días), se calcula el Sharpe ratio rolling de 21 días para cada ticker del universo vanilla. Se seleccionan los 5 con mejor Sharpe (momentum relativo). Cada slot que no supera el Sharpe de SHY se reemplaza por SHY (momentum absoluto — la parte "dual"). Pesos iguales entre los 5 slots.

Ref: MScFE SKILL §6.1 (rolling features), §8.2 (Trap 5 — no look-ahead)

In [10]:
STRAT005_ID = 'STRAT005'

dual_mom_universe = [
    'SHY', 'BOXX', 'TLT', 'IEF', 'AGG', 'LQD', 'HYG', 'EMB', 'TIP', 'PFF',
    'SPY', 'EFA', 'VEA', 'EEM', 'VNQ', 'GLD', 'SLV', 'USO', 'DBC', 'DBMF',
    'BTC-USD', 'VT',
]
dual_mom_available = [t for t in dual_mom_universe if t in returns_daily.columns]
print(f'STRAT-005 universo disponible: {len(dual_mom_available)}/{len(dual_mom_universe)} tickers')
if len(dual_mom_available) < len(dual_mom_universe):
    missing = [t for t in dual_mom_universe if t not in returns_daily.columns]
    print(f'  [warn] Faltantes: {missing}')

DUAL_MOM_LOOKBACK   = 21
DUAL_MOM_REBAL_DAYS = 21
DUAL_MOM_N_SELECT   = 5
DUAL_MOM_RF_TICKER  = 'SHY'

sharpe_panel_strat005 = rolling_sharpe_matrix(
    returns_daily[dual_mom_available],
    lookback=DUAL_MOM_LOOKBACK,
    rf_annual=rf_annual,
    periods=TRADING_DAYS,
)

sharpe_output_path = SIGNALS / 'sharpe_ratios_STRAT005.csv'
sharpe_panel_strat005.to_csv(sharpe_output_path)
print(f'Sharpe ratios guardados: {sharpe_output_path}')
print(f'Última fila ({sharpe_panel_strat005.index.max().date()}):')
print(sharpe_panel_strat005.iloc[-1].sort_values(ascending=False).round(2).to_string())

weights_strat005 = select_dual_momentum(
    sharpe_panel_strat005,
    rf_ticker=DUAL_MOM_RF_TICKER,
    n_select=DUAL_MOM_N_SELECT,
    rebal_days_signal=DUAL_MOM_REBAL_DAYS,
)

returns_strat005 = build_strategy_returns(
    system_tickers=dual_mom_available,
    hedge_map={},
    weights_df=weights_strat005,
    signal_fn=signal_always_long,
    costs_bps=10.0,
    rebal_days=DUAL_MOM_REBAL_DAYS,
)

output_path_005 = SIGNALS / f'daily_returns_{STRAT005_ID}.csv'
returns_strat005.to_frame('daily_return').to_csv(output_path_005)
print(f'\nSTRAT-005 guardado: {output_path_005}')
print(f'Período: {returns_strat005.index.min().date()} → {returns_strat005.index.max().date()}')
print(f'Retorno medio diario: {returns_strat005.mean():.4%}')

active = weights_strat005.iloc[-1]
active = active[active > 0]
print(f'\nÚltima selección activa ({weights_strat005.index[-1].date()}):')
for t, w in active.items():
    print(f'  {t}: {w:.0%}')

STRAT-005 universo disponible: 17/22 tickers
  [warn] Faltantes: ['USO', 'DBC', 'DBMF', 'BTC-USD', 'VT']
Sharpe ratios guardados: C:\Users\Sebastián\portfolio_quant\data\signals\sharpe_ratios_STRAT005.csv
Última fila (2026-06-24):
TLT     4.71
EMB     3.34
LQD     2.77
AGG     2.72
IEF     2.44
EEM     0.88
HYG     0.85
SHY     0.48
VEA     0.17
BOXX    0.15
VNQ     0.14
TIP     0.12
EFA    -0.08
SPY    -1.06
PFF    -2.18
GLD    -4.91
SLV    -5.99



STRAT-005 guardado: C:\Users\Sebastián\portfolio_quant\data\signals\daily_returns_STRAT005.csv
Período: 2016-06-28 → 2026-06-24
Retorno medio diario: 0.0449%

Última selección activa (2026-06-24):
  BOXX: 20%
  TLT: 20%
  EMB: 20%
  SPY: 20%
  VNQ: 20%


## STRAT-003 — Trend Following puro (señal binaria)


In [11]:
STRAT003_ID = 'STRAT003'

if 'SPY' not in prices_adjusted.columns:
    print('[SKIP] SPY no disponible en el panel')
else:
    weights_strat003 = compute_position_sizing_equal(['SPY'], prices_adjusted)

    returns_strat003 = build_strategy_returns(
        system_tickers=['SPY'],
        hedge_map={},
        weights_df=weights_strat003,
        signal_fn=signal_ma_binary,
        collateral_tickers=['BOXX', 'SHY'],
        costs_bps=5.0,
        rebal_days=1,
        ma_window=ma_length,
    )

    output_path_003 = SIGNALS / f'daily_returns_{STRAT003_ID}.csv'
    returns_strat003.to_frame('daily_return').to_csv(output_path_003)
    print(f'STRAT-003 guardado: {output_path_003}')
    print(f'Retorno medio diario: {returns_strat003.mean():.4%}')

STRAT-003 guardado: C:\Users\Sebastián\portfolio_quant\data\signals\daily_returns_STRAT003.csv
Retorno medio diario: 0.0093%
